    1. Import library untuk manipulasi data (pandas), tokenisasi teks (word_tokenize), dan daftar stopword bahasa Indonesia.

    2. sw_indo = gabungan stopword Indonesia (kata seperti "yang", "di", "ke", dll) plus semua tanda baca (., ,, ", dll). Ini nanti dipakai supaya kata-kata umum dan tanda baca diabaikan saat perhitungan TF-IDF.

In [18]:
import pandas as pd
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from string import punctuation


sw_indo = stopwords.words("indonesian") + list(punctuation)

# Import DATA


In [19]:
df = pd.read_csv("data/kompas.csv")
df.head()

,teks
0,Ginandjar Tetap Ditahan. Jaksa Agung Dilaporka...
1,Jakarta Dikangkangi Para Preman\nKALAU tak pun...
2,Penyimpangan di Setpres Seolah Terjadi Sekaran...
3,"Dibayarkan, Rapel Kenaikan Gaji Pegawai Pos\nK..."
4,"Stop Kekerasan, Elite agar Duduk Bersama\nSeju..."


# Extract TFIDF

In [20]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [21]:
tfidf = TfidfVectorizer(ngram_range=(1,2), tokenizer=word_tokenize,stop_words=sw_indo)
tfidf_matrix =  tfidf.fit_transform(df.teks)


#  fit_transform, supaya vocab ada di scope global
vocab = tfidf.get_feature_names_out()

c:\Users\ASUS\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\feature_extraction\text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
c:\Users\ASUS\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\feature_extraction\text.py:402: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['``'] not in stop_words.
  warnings.warn(


    1. Membuat objek TfidfVectorizer yang akan mengubah teks menjadi representasi angka (vektor) berdasarkan skor TF-IDF (Term Frequency–Inverse Document Frequency — mengukur seberapa penting sebuah kata di suatu dokumen dibanding seluruh dokumen).

    2. ngram_range=(1,2) → mempertimbangkan kata tunggal (unigram) dan pasangan dua kata berurutan (bigram), misal "korupsi" dan "kasus korupsi".

    3. tokenizer=word_tokenize → cara memecah kalimat jadi kata-kata pakai NLTK.

    4. stop_words=sw_indo → kata-kata umum dan tanda baca diabaikan.
    fit_transform → melatih vectorizer di seluruh data teks dan langsung mengubah semua dokumen jadi matriks TF-IDF (tfidf_matrix), berukuran 2008 dokumen x 548.426 fitur (kata/frasa unik).

    5. Warning yang muncul (token_pattern tidak dipakai, ada token ` `` yang bukan stopword) itu wajar, bisa diabaikan.

# TFIDF similarty -> Document Similarty(kemiripan)

In [22]:
from sklearn.metrics.pairwise import cosine_similarity

In [23]:
sim = cosine_similarity(tfidf_matrix[0], tfidf_matrix)
sim

array([[1.        , 0.00858328, 0.01060043, ..., 0.00856287, 0.00677808,
        0.01513341]], shape=(1, 2008))

    1. Menghitung cosine similarity antara dokumen ke-0 dengan semua dokumen lain (termasuk dirinya sendiri).

    2. Cosine similarity mengukur kemiripan arah dua vektor; nilainya 0 (tidak mirip) sampai 1 (identik).

    3. Hasilnya array 1x2008: skor kemiripan dokumen 0 terhadap dokumen 0, 1, 2, ..., 2007. Nilai pertama 1.0 karena dokumen dibandingkan dengan dirinya sendiri.

index nya yang di sort

 yang kanan paling mirip sedangakan yang kiri tidak mirip

In [24]:
sim.argsort() 


array([[1131,  932, 1593, ...,  215,  144,    0]], shape=(1, 2008))

In [25]:
df.teks[0][:200]

'Ginandjar Tetap Ditahan. Jaksa Agung Dilaporkan ke Polri\nKejaksaan Agung memutuskan untuk tetap menahan tersangka kasus korupsi, Ginandjar Kartasasmita, sampai batas waktu yang ditentukan KUHAP. Sedan'

In [26]:
df.teks[144][:200]

'Kejaksaan Agung Terbitkan Surat Penahanan Baru\nKejaksaan Agung (Kejagung) akhirnya menerbitkan surat perintah penahanan yang baru terhadap mantan Menteri Pertambangan dan Energi Ginandjar Kartasasmita'

In [27]:
df.teks[1131][:200]

'Merpati Mendarat Darurat\n\nPesawat Penumpang CASA 212 Merpati Ambon-Saumlaki, Kamis (18/4) pukul 16.30, mendarat darurat di Desa Rumah Lewang Besar, Kecamatan Pulau-pulau Babar, Kabupaten Maluku Tengga'

# Keyword Extraction 

In [28]:
tfidf_matrix[0].toarray() # konvert ke array dulu

array([[0.02115058, 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ]], shape=(1, 548426))

In [29]:
sorted_tfidf = tfidf_matrix[0].toarray()[0].argsort()
sorted_tfidf


array([548416,     17,     18, ..., 386379, 436652, 169219],
      shape=(548426,))

In [30]:
vocab[169219]   # sekarang tidak error

'ginandjar'

In [31]:
vocab[23452]

'46 juta'

In [32]:
[vocab[idx] for idx in reversed(sorted_tfidf[-10:])]

['ginandjar',
 'putusan',
 'penahanan',
 'hukum ginandjar',
 'kuasa hukum',
 'rusman',
 'kejaksaan',
 'hakim rusman',
 'kuasa',
 '9 april']

In [33]:
def extract_keywords_tfidf(doc, tfidf, topk=10):
    matrix = tfidf.transform([doc])
    vocab = tfidf.get_feature_names_out()  # Perubahan dari get_feature_names() ke get_feature_names_out()

    sorted_tfidf = matrix[0].toarray()[0].argsort()
    return [vocab[idx] for idx in reversed(sorted_tfidf[-topk:])]

In [34]:
text = """
Gempa bumi berkekuatan 6,5 magnitudo mengguncang wilayah Jawa Barat pada Jumat pagi. Badan Meteorologi, Klimatologi, dan Geofisika (BMKG) melaporkan bahwa gempa terjadi pada pukul 07.45 WIB dengan pusat gempa berada di kedalaman 25 km di bawah laut, sekitar 80 km barat daya Kabupaten Sukabumi.

Masyarakat di beberapa daerah seperti Bandung, Bogor, dan Jakarta merasakan guncangan yang cukup kuat selama 5-10 detik. Sejumlah bangunan dilaporkan mengalami kerusakan ringan, namun hingga saat ini belum ada laporan korban jiwa.

"Kami masih terus memantau perkembangan situasi dan mengimbau masyarakat untuk tetap tenang serta waspada terhadap kemungkinan gempa susulan," ujar Kepala BMKG, Dr. Rina Widiyanti.

Sementara itu, Badan Nasional Penanggulangan Bencana (BNPB) telah mengerahkan tim tanggap darurat ke daerah terdampak untuk menilai kerusakan serta memberikan bantuan bagi warga yang terdampak.

BMKG memastikan bahwa gempa ini tidak berpotensi tsunami, namun masyarakat yang berada di pesisir pantai diimbau untuk tetap waspada terhadap potensi gempa susulan.

Pemerintah daerah setempat mengimbau masyarakat untuk selalu mengikuti informasi resmi dari BMKG dan tidak mudah percaya dengan berita hoaks yang beredar di media sosial."""



    1. Ini fungsi umum untuk ekstrak keyword dari teks baru apa pun (doc), bukan cuma dokumen yang sudah ada di df.

    2. tfidf.transform([doc]) → mengubah teks baru itu jadi vektor TF-IDF menggunakan vocabulary yang sudah dipelajari sebelumnya (bukan fit_transform, karena kita tidak mau melatih ulang vectorizer-nya, cukup pakai yang sudah ada).

    3. Di dalam fungsi ini, vocab didefinisikan secara lokal — jadi fungsi ini sendiri tidak akan error saat dipanggil. Tapi variabel vocab di sini tidak "keluar" ke luar fungsi.

    4. topk=10 → default ambil 10 kata teratas.

    5. Fungsi mengembalikan daftar kata-kata dengan skor TF-IDF tertinggi di teks tersebut, alias keyword dari teks itu.

In [35]:
extract_keywords_tfidf(text, tfidf)

['gempa',
 'gempa susulan',
 'mengimbau masyarakat',
 'susulan',
 'waspada',
 'kerusakan',
 'mengimbau',
 'bogor jakarta',
 'gempa 07.45',
 'merasakan guncangan']

    Memanggil fungsi di atas dengan teks gempa tadi → akan menghasilkan kata-kata seperti "gempa", "bmkg", "magnitudo", dll (kata-kata paling khas di teks itu berdasarkan pembobotan TF-IDF dari korpus berita Kompas).



In [36]:
tfidf.idf_.argsort() # paling langka

array([ 28068,      0, 200992, ..., 548424, 548425,      2],
      shape=(548426,))

    1. tfidf.idf_ adalah nilai IDF (Inverse Document Frequency) untuk setiap kata di vocabulary — makin jarang sebuah kata muncul di seluruh dokumen, makin tinggi nilai IDF-nya.

    2. argsort() mengurutkan index kata dari IDF terkecil (kata sering muncul, kurang unik) ke terbesar (kata sangat jarang muncul, paling unik/langka di seluruh korpus).

In [37]:
vocab[548425]

'zx diserbu'

    Mengecek kata dengan index IDF tertinggi (index terakhir dari hasil argsort() di atas) → yaitu kata paling langka di seluruh 2008 dokumen berita